# E2E UniversalXAS with weighted shell features

Use the frozen encoder from `fullTrainingCopy072726/m3gnetAll8E2EUniversal/e2e_universal_seed42` to export 192D features:

- absorbing-site state: 64D;
- inverse-distance-weighted 0–3 Å shell: 64D;
- inverse-distance-weighted 3–5 Å shell: 64D.

Train UniversalXAS with the later validation-selected plateau settings, then fine-tune one head per FEFF element with the later best fine-tuning settings. The final table contains test eta only.


## 1. Setup and data validation


In [1]:
from __future__ import annotations

import hashlib
import json
import os
import random
import sys
from pathlib import Path

import dgl
import lightning.pytorch as pl
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader

from omnixas.data.ml_data import MLData, MLSplits
from omnixas.model.xasblock import XASBlock
from omnixas.model.xasblock_regressor import XASBlockRegressor

REPO_ROOT = Path.cwd().resolve()
while not ((REPO_ROOT / "pyproject.toml").exists() and (REPO_ROOT / "omnixas").is_dir()):
    if REPO_ROOT.parent == REPO_ROOT:
        raise FileNotFoundError("Run this notebook from inside the OmniXAS repository")
    REPO_ROOT = REPO_ROOT.parent

sys.path.insert(0, str(REPO_ROOT / "tutorial_omnixas"))
from train_all8_feff import (  # noqa: E402
    BATCH,
    ENCODER_BATCH,
    FEATURE_SCALE,
    HEAD_WIDTHS,
    INPUT_DIM,
    OUTPUT_DIM,
    SPLITS,
    CollateGraphs,
    EncoderModel,
    FEFFDataset,
    checkpoint_score,
    compute_theta_and_phi,
    create_line_graph,
    patch_matgl_gpu_constants,
    polynomial_cutoff,
    torch_load,
)
from omnixas.model.training import PlModule  # noqa: E402

FEFF_TASKS = ["Ti_FEFF", "V_FEFF", "Cr_FEFF", "Mn_FEFF", "Fe_FEFF", "Co_FEFF", "Ni_FEFF", "Cu_FEFF"]
SHELL_INNER, SHELL_OUTER = 3.0, 5.0
FEATURE_DIM = 3 * INPUT_DIM
NUM_WORKERS = 4
GNN_DROPOUT = 0.10
MIN_LR = 1e-4
ETA_MIN = 1e-6

SOURCE_RUN = Path(os.environ.get(
    "OMNIXAS_E2E_UNIVERSAL_RUN",
    REPO_ROOT.parent / "fullTrainingCopy072726" / "m3gnetAll8E2EUniversal" / "e2e_universal_seed42",
)).expanduser().resolve()
CUSTOM_ENCODER_CKPT = SOURCE_RUN / "best_e2e_encoder_universal.ckpt"
RAW_ROOT = Path(os.environ.get("OMNIXAS_DATA_ROOT", REPO_ROOT.parent / "OmniXAS_data")).expanduser().resolve() / "materialscloud_omnixas_raw" / "extracted"
PREVIOUS_RESULTS = SOURCE_RUN / "notebook_new_universal_runs" / "analysis" / "new_universal_feff"
OUT_ROOT = SOURCE_RUN / "notebook_shell_plateau_runs" / "weighted_shell_3_5"
FEATURE_DIR = OUT_ROOT / "features"
HEAD_ROOT = OUT_ROOT / "heads"

UNIVERSAL = {
    "name": "plateau_lr5e-4_do010_seed44", "seed": 44, "dropout": 0.10,
    "lr": 5e-4, "scheduler": "plateau", "epochs": 800, "patience": 60,
    "batch_size": 32, "plateau_factor": 0.5, "plateau_patience": 8,
    "plateau_min_lr": 1e-6,
}
TUNED = {
    "name": "cosT500_lr3e-4_do010_seed145", "seed": 145, "dropout": 0.10,
    "lr": 3e-4, "scheduler": "cosine", "epochs": 1000, "patience": 25,
    "cosine_t": 500,
}

required = [
    CUSTOM_ENCODER_CKPT,
    RAW_ROOT,
    PREVIOUS_RESULTS / "selected_universal_test_by_dataset.csv",
    PREVIOUS_RESULTS / "tuned_feff_selected_by_validation.csv",
]
missing = [str(path) for path in required if not path.exists()]
if missing:
    raise FileNotFoundError(f"Missing required input(s): {missing}")

DATA_DIR = REPO_ROOT / "tutorial_omnixas" / "ml_data"
ID_DIR = REPO_ROOT / "tutorial_omnixas" / "material_id_and_site"
OUT_ROOT.mkdir(parents=True, exist_ok=True)
DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
torch.set_float32_matmul_precision("medium")
patch_matgl_gpu_constants()

split_ids = {
    task: {split: [row for row in (ID_DIR / f"{task}_{split}.txt").read_text().splitlines() if row] for split in SPLITS}
    for task in FEFF_TASKS
}
y_true = {
    task: {split: np.atleast_2d(np.loadtxt(DATA_DIR / f"{task}_{split}_y.txt", dtype=np.float32)) for split in SPLITS}
    for task in FEFF_TASKS
}
for task in FEFF_TASKS:
    for split in SPLITS:
        if y_true[task][split].shape != (len(split_ids[task][split]), OUTPUT_DIM):
            raise ValueError(f"{task} {split}: ID/target row mismatch")
        if not np.isfinite(y_true[task][split]).all():
            raise ValueError(f"{task} {split}: non-finite targets")
    materials = {split: {row.rsplit("_", 1)[0] for row in split_ids[task][split]} for split in SPLITS}
    for left, right in (("train", "val"), ("train", "test"), ("val", "test")):
        overlap = materials[left] & materials[right]
        if overlap:
            raise ValueError(f"Material leakage in {task} {left}/{right}: {sorted(overlap)[:5]}")


## 2. Frozen encoder provenance


In [2]:
PAPER_FEFF_ETA = {
    ("Ti_FEFF", "UniversalXAS"): 4.19, ("Ti_FEFF", "Tuned-UniversalXAS"): 7.63,
    ("V_FEFF", "UniversalXAS"): 5.19, ("V_FEFF", "Tuned-UniversalXAS"): 9.22,
    ("Cr_FEFF", "UniversalXAS"): 7.13, ("Cr_FEFF", "Tuned-UniversalXAS"): 10.44,
    ("Mn_FEFF", "UniversalXAS"): 13.15, ("Mn_FEFF", "Tuned-UniversalXAS"): 29.81,
    ("Fe_FEFF", "UniversalXAS"): 6.04, ("Fe_FEFF", "Tuned-UniversalXAS"): 8.98,
    ("Co_FEFF", "UniversalXAS"): 9.58, ("Co_FEFF", "Tuned-UniversalXAS"): 19.83,
    ("Ni_FEFF", "UniversalXAS"): 6.43, ("Ni_FEFF", "Tuned-UniversalXAS"): 11.21,
    ("Cu_FEFF", "UniversalXAS"): 2.75, ("Cu_FEFF", "Tuned-UniversalXAS"): 4.81,
}

digest = hashlib.sha256()
with CUSTOM_ENCODER_CKPT.open("rb") as checkpoint_file:
    for chunk in iter(lambda: checkpoint_file.read(1024 * 1024), b""):
        digest.update(chunk)
CHECKPOINT_SHA256 = digest.hexdigest()


## 3. Weighted-shell collator and pooling


In [3]:
class WeightedShellCollator:
    def __init__(self, encoder):
        self.graph_collator = CollateGraphs(encoder)

    def __call__(self, batch):
        graphs, sites, spectra = [], [], []
        shell_rows, shell_sources, shell_distances, shell_ids = [], [], [], []
        node_offset = 0
        for row_index, (_task, structure, site, spectrum) in enumerate(batch):
            graph = self.graph_collator.graph(structure)
            graphs.append(graph)
            sites.append(node_offset + site)
            spectra.append(spectrum)
            center = structure[site].coords
            for neighbor in structure.get_sites_in_sphere(center, SHELL_OUTER, include_index=True):
                distance = float(getattr(neighbor, "nn_distance", neighbor.distance(structure[site])))
                if distance <= 1e-6:
                    continue
                if distance <= SHELL_INNER:
                    shell_id = 0
                elif distance <= SHELL_OUTER:
                    shell_id = 1
                else:
                    continue
                shell_rows.append(row_index)
                shell_sources.append(node_offset + int(neighbor.index))
                shell_distances.append(distance)
                shell_ids.append(shell_id)
            node_offset += graph.num_nodes()
        return {
            "graph": dgl.batch(graphs),
            "site": torch.tensor(sites, dtype=torch.long),
            "y": torch.stack(spectra).float(),
            "shell_row": torch.tensor(shell_rows, dtype=torch.long),
            "shell_source": torch.tensor(shell_sources, dtype=torch.long),
            "shell_distance": torch.tensor(shell_distances, dtype=torch.float32),
            "shell_id": torch.tensor(shell_ids, dtype=torch.long),
        }

    def pool_weighted_shells(self, node, batch):
        n_sites = len(batch["site"])
        pooled = node.new_zeros(n_sites, 2, INPUT_DIM)
        rows = batch["shell_row"].to(node.device)
        sources = batch["shell_source"].to(node.device)
        distances = batch["shell_distance"].to(node.device)
        shell_ids = batch["shell_id"].to(node.device)
        for shell_id in (0, 1):
            mask = shell_ids == shell_id
            if not mask.any():
                continue
            shell_rows = rows[mask]
            weights = 1.0 / distances[mask].clamp_min(1e-6)
            norm = node.new_zeros(n_sites).scatter_add(0, shell_rows, weights).clamp_min(1e-12)
            pooled[:, shell_id, :].index_add_(0, shell_rows, (weights / norm[shell_rows]).unsqueeze(-1) * node[sources[mask]])
        return torch.cat([node[batch["site"].to(node.device)], pooled[:, 0, :], pooled[:, 1, :]], dim=1)


## 4. Weighted-shell feature export


In [4]:
FEATURE_CONFIG = {
    "custom_encoder_checkpoint": str(CUSTOM_ENCODER_CKPT),
    "custom_encoder_sha256": CHECKPOINT_SHA256,
    "feature_scale": FEATURE_SCALE,
    "shell_inner": SHELL_INNER,
    "shell_outer": SHELL_OUTER,
    "encoder_graph_cutoff": 4.0,
    "feature_dim": FEATURE_DIM,
}
FEATURE_DIR.mkdir(parents=True, exist_ok=True)
feature_config_path = FEATURE_DIR / "feature_config.json"
if feature_config_path.exists():
    if json.loads(feature_config_path.read_text(encoding="utf-8")) != FEATURE_CONFIG:
        raise ValueError(f"Feature provenance changed; use a new OUT_ROOT: {feature_config_path}")
else:
    feature_config_path.write_text(json.dumps(FEATURE_CONFIG, indent=2) + "\n", encoding="utf-8")


missing = []
for task in FEFF_TASKS:
    for split in SPLITS:
        x_path = FEATURE_DIR / f"{task}_{split}_X.txt"
        y_path = FEATURE_DIR / f"{task}_{split}_y.txt"
        if not x_path.exists() or not y_path.exists():
            missing.append((task, split))
            continue
        X = np.atleast_2d(np.loadtxt(x_path, dtype=np.float32))
        y = np.atleast_2d(np.loadtxt(y_path, dtype=np.float32))
        if X.shape != (len(split_ids[task][split]), FEATURE_DIM) or y.shape != y_true[task][split].shape:
            raise ValueError(f"Invalid cached features for {task} {split}: X={X.shape} y={y.shape}")
        if not np.isfinite(X).all() or not np.isfinite(y).all():
            raise ValueError(f"Non-finite cached features for {task} {split}")
if missing:
    encoder_model = EncoderModel(dropout=GNN_DROPOUT)
    checkpoint = torch_load(CUSTOM_ENCODER_CKPT)
    state = checkpoint.get("state_dict")
    if state is None:
        raise ValueError(f"Checkpoint has no state_dict: {CUSTOM_ENCODER_CKPT}")
    encoder_state = {k.removeprefix("model.encoder."): v for k, v in state.items() if k.startswith("model.encoder.")}
    if not encoder_state:
        raise ValueError(f"Checkpoint has no model.encoder parameters: {CUSTOM_ENCODER_CKPT}")
    encoder_model.encoder.load_state_dict(encoder_state, strict=True)
    encoder = encoder_model.encoder.to(DEVICE).eval()
    for parameter in encoder.parameters():
        parameter.requires_grad = False
    collate = WeightedShellCollator(encoder)
    with torch.inference_mode():
        for task, split in missing:
            loader = DataLoader(
                FEFFDataset(REPO_ROOT, RAW_ROOT, [task], split),
                batch_size=ENCODER_BATCH,
                shuffle=False,
                collate_fn=collate,
                num_workers=NUM_WORKERS,
            )
            Xs, ys = [], []
            for batch in loader:
                graph = batch["graph"].to(DEVICE)
                graph.edata["rbf"] = encoder.bond_expansion(graph.edata["bond_dist"])
                line_graph = create_line_graph(graph.to("cpu"), encoder.threebody_cutoff).to(graph.device)
                line_graph.apply_edges(compute_theta_and_phi)
                basis = encoder.basis_expansion(line_graph)
                cutoff = polynomial_cutoff(graph.edata["bond_dist"], encoder.threebody_cutoff)
                node, edge, state = encoder.embedding(graph.ndata["node_type"], graph.edata["rbf"], None)
                for three_body, graph_layer in zip(encoder.three_body_interactions, encoder.graph_layers, strict=True):
                    edge = three_body(graph, line_graph, basis, cutoff, node, edge)
                    edge, node, state = graph_layer(graph, edge, node, state)
                features = collate.pool_weighted_shells(node, batch)
                Xs.append((features * FEATURE_SCALE).cpu().numpy())
                ys.append(batch["y"].numpy())
            X, y = np.concatenate(Xs), np.concatenate(ys)
            if X.shape != (len(split_ids[task][split]), FEATURE_DIM) or y.shape != y_true[task][split].shape:
                raise RuntimeError(f"Invalid export for {task} {split}: X={X.shape} y={y.shape}")
            np.savetxt(FEATURE_DIR / f"{task}_{split}_X.txt", X)
            np.savetxt(FEATURE_DIR / f"{task}_{split}_y.txt", y)
            print(f"exported {task} {split}: X={X.shape} y={y.shape}")
    del encoder
    del encoder_model
    torch.cuda.empty_cache()
else:
    print("all shell feature splits cached")


exported Ti_FEFF train: X=(5140, 192) y=(5140, 141)
exported Ti_FEFF val: X=(641, 192) y=(641, 141)
exported Ti_FEFF test: X=(641, 192) y=(641, 141)
exported V_FEFF train: X=(8653, 192) y=(8653, 141)
exported V_FEFF val: X=(1080, 192) y=(1080, 141)
exported V_FEFF test: X=(1080, 192) y=(1080, 141)
exported Cr_FEFF train: X=(2457, 192) y=(2457, 141)
exported Cr_FEFF val: X=(305, 192) y=(305, 141)
exported Cr_FEFF test: X=(305, 192) y=(305, 141)
exported Mn_FEFF train: X=(13644, 192) y=(13644, 141)
exported Mn_FEFF val: X=(1704, 192) y=(1704, 141)
exported Mn_FEFF test: X=(1704, 192) y=(1704, 141)
exported Fe_FEFF train: X=(9657, 192) y=(9657, 141)
exported Fe_FEFF val: X=(1205, 192) y=(1205, 141)
exported Fe_FEFF test: X=(1205, 192) y=(1205, 141)
exported Co_FEFF train: X=(8605, 192) y=(8605, 141)
exported Co_FEFF val: X=(1074, 192) y=(1074, 141)
exported Co_FEFF test: X=(1074, 192) y=(1074, 141)
exported Ni_FEFF train: X=(3471, 192) y=(3471, 141)
exported Ni_FEFF val: X=(432, 192) y=(4

## 5. UniversalXAS training and evaluation


In [5]:
feff_splits = {}
for task in FEFF_TASKS:
    split_data = {}
    for split in SPLITS:
        split_data[split] = MLData(
            X=np.atleast_2d(np.loadtxt(FEATURE_DIR / f"{task}_{split}_X.txt", dtype=np.float32)),
            y=np.atleast_2d(np.loadtxt(FEATURE_DIR / f"{task}_{split}_y.txt", dtype=np.float32)),
        )
    feff_splits[task] = MLSplits(**split_data)

universal_split = MLSplits(
    train=MLData(X=np.concatenate([feff_splits[t].train.X for t in FEFF_TASKS]), y=np.concatenate([feff_splits[t].train.y for t in FEFF_TASKS])),
    val=MLData(X=np.concatenate([feff_splits[t].val.X for t in FEFF_TASKS]), y=np.concatenate([feff_splits[t].val.y for t in FEFF_TASKS])),
    test=MLData(X=np.concatenate([feff_splits[t].test.X for t in FEFF_TASKS]), y=np.concatenate([feff_splits[t].test.y for t in FEFF_TASKS])),
)

universal_dir = HEAD_ROOT / "universalXAS" / "All_FEFF" / "runs" / UNIVERSAL["name"]
universal_ckpt = None
if universal_dir.exists():
    universal_checkpoints = sorted(universal_dir.glob("best*.ckpt"))
    if universal_checkpoints:
        universal_ckpt = min(universal_checkpoints, key=checkpoint_score)
if universal_ckpt is None:
    if universal_dir.exists() and any(universal_dir.iterdir()):
        raise RuntimeError(f"Incomplete UniversalXAS run: {universal_dir}")
    pl.seed_everything(UNIVERSAL["seed"], workers=True)
    random.seed(UNIVERSAL["seed"])
    np.random.seed(UNIVERSAL["seed"])
    torch.manual_seed(UNIVERSAL["seed"])
    XASBlock.DROPOUT = UNIVERSAL["dropout"]
    universal_model = XASBlockRegressor(
        directory=str(universal_dir), overwrite_save_dir=False,
        input_dim=FEATURE_DIM, output_dim=OUTPUT_DIM, hidden_dims=HEAD_WIDTHS,
        batch_size=UNIVERSAL["batch_size"], max_epochs=UNIVERSAL["epochs"],
        early_stopping_patience=UNIVERSAL["patience"], initial_lr=UNIVERSAL["lr"],
        min_lr=MIN_LR, use_lr_finder=False, use_early_stopping=True,
        monitor_metric="val_median_mse", shuffle=True,
        lr_scheduler=UNIVERSAL["scheduler"],
        cosine_t_max=UNIVERSAL.get("cosine_t") or UNIVERSAL["epochs"],
        cosine_eta_min=ETA_MIN, warmup_epochs=10,
        plateau_factor=UNIVERSAL.get("plateau_factor", 0.5),
        plateau_patience=UNIVERSAL.get("plateau_patience", 5),
        plateau_min_lr=UNIVERSAL.get("plateau_min_lr", ETA_MIN),
    )
    universal_model.fit(universal_split)
    universal_ckpt = Path(universal_model.cfg.fetch_checkpoint("best"))
else:
    print("UniversalXAS cached:", universal_ckpt)

universal_rows = []
for task in FEFF_TASKS:
    split = feff_splits[task]
    data = split.test
    universal_module = PlModule.load_from_checkpoint(
        checkpoint_path=str(universal_ckpt),
        model=XASBlock(FEATURE_DIM, HEAD_WIDTHS, OUTPUT_DIM), lr=1e-4,
    ).to(DEVICE).eval()
    predictions = []
    X = data.X.astype(np.float32, copy=False)
    with torch.no_grad():
        for start in range(0, len(X), 1024):
            xb = torch.as_tensor(X[start : start + 1024], dtype=torch.float32, device=DEVICE)
            predictions.append(universal_module(xb).cpu().numpy())
    pred = np.concatenate(predictions)
    test_mse = float(np.median(np.mean((data.y - pred) ** 2, axis=1)))
    baseline = data.y - split.train.y.mean(axis=0, keepdims=True)
    test_baseline_mse = float(np.median(np.mean(baseline ** 2, axis=1)))
    universal_rows.append({
        "dataset": task, "test_median_mse": test_mse,
        "test_baseline_median_mse": test_baseline_mse,
        "test_eta": test_baseline_mse / test_mse,
    })
universal_df = pd.DataFrame(universal_rows)
universal_df.to_csv(OUT_ROOT / "new_universal_eval.csv", index=False)


Seed set to 44
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
/home/anton/miniconda3/envs/omnixas/lib/python3.11/site-packages/lightning/pytorch/callbacks/model_checkpoint.py:881: Checkpoint directory /mnt/c/Users/anton/desktop/fullTrainingCopy072726/m3gnetAll8E2EUniversal/e2e_universal_seed42/notebook_shell_plateau_runs/weighted_shell_3_5/heads/universalXAS/All_FEFF/runs/plateau_lr5e-4_do010_seed44 exists and is not empty.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name  | Type     | Params | Mode  | FLOPs
---------------------------------------------------
0 | loss  | MSELoss  | 0      | train | 0    
1 | model | XASBlock | 703 K  | train | 0    
---------------------------------------------------
703 K     Trainable para

Sanity Checking: |                                                                                | 0/? [00:00…

/home/anton/miniconda3/envs/omnixas/lib/python3.11/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=31` in the `DataLoader` to improve performance.
/home/anton/miniconda3/envs/omnixas/lib/python3.11/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=31` in the `DataLoader` to improve performance.


Training: |                                                                                       | 0/? [00:00…

Validation: |                                                                                     | 0/? [00:00…

...

Validation: |                                                                                     | 0/? [00:00…

Validation: |                                                                                     | 0/? [00:00…

Validation: |                                                                                     | 0/? [00:00…

Validation: |                                                                                     | 0/? [00:00…

2026-07-29 13:27:08.347 | INFO     | omnixas.model.xasblock_regressor:fit:246 - Best models saved at /mnt/c/Users/anton/desktop/fullTrainingCopy072726/m3gnetAll8E2EUniversal/e2e_universal_seed42/notebook_shell_plateau_runs/weighted_shell_3_5/heads/universalXAS/All_FEFF/runs/plateau_lr5e-4_do010_seed44/best-model-epoch=273-val_median_mse=0.0021.ckpt
2026-07-29 13:27:08.349 | INFO     | omnixas.model.xasblock_regressor:fit:247 - Best validation loss: 0.004855489358305931


## 6. Tuned-UniversalXAS training and evaluation


In [6]:
tuned_rows = []
for task in FEFF_TASKS:
    split = feff_splits[task]
    tuned_dir = HEAD_ROOT / "tunedUniversalXAS" / task / "runs" / TUNED["name"]
    tuned_ckpt = None
    if tuned_dir.exists():
        tuned_checkpoints = sorted(tuned_dir.glob("best*.ckpt"))
        if tuned_checkpoints:
            tuned_ckpt = min(tuned_checkpoints, key=checkpoint_score)
    if tuned_ckpt is None:
        if tuned_dir.exists() and any(tuned_dir.iterdir()):
            raise RuntimeError(f"Incomplete Tuned-UniversalXAS run: {tuned_dir}")
        pl.seed_everything(TUNED["seed"], workers=True)
        random.seed(TUNED["seed"])
        np.random.seed(TUNED["seed"])
        torch.manual_seed(TUNED["seed"])
        XASBlock.DROPOUT = TUNED["dropout"]
        tuned_model = XASBlockRegressor(
            directory=str(universal_ckpt.parent), overwrite_save_dir=False,
            input_dim=FEATURE_DIM, output_dim=OUTPUT_DIM, hidden_dims=HEAD_WIDTHS,
            batch_size=BATCH[task], max_epochs=TUNED["epochs"],
            early_stopping_patience=TUNED["patience"], initial_lr=TUNED["lr"],
            min_lr=MIN_LR, use_lr_finder=False, use_early_stopping=True,
            monitor_metric="val_median_mse", shuffle=True,
            lr_scheduler=TUNED["scheduler"],
            cosine_t_max=TUNED.get("cosine_t") or TUNED["epochs"],
            cosine_eta_min=ETA_MIN, warmup_epochs=10,
            plateau_factor=TUNED.get("plateau_factor", 0.5),
            plateau_patience=TUNED.get("plateau_patience", 5),
            plateau_min_lr=TUNED.get("plateau_min_lr", ETA_MIN),
        )
        tuned_model.load("best")
        tuned_model.cfg.directory = str(tuned_dir)
        tuned_model.fit(split)
        tuned_ckpt = Path(tuned_model.cfg.fetch_checkpoint("best"))

    data = split.test
    tuned_module = PlModule.load_from_checkpoint(
        checkpoint_path=str(tuned_ckpt),
        model=XASBlock(FEATURE_DIM, HEAD_WIDTHS, OUTPUT_DIM), lr=1e-4,
    ).to(DEVICE).eval()
    predictions = []
    X = data.X.astype(np.float32, copy=False)
    with torch.no_grad():
        for start in range(0, len(X), 1024):
            xb = torch.as_tensor(X[start : start + 1024], dtype=torch.float32, device=DEVICE)
            predictions.append(tuned_module(xb).cpu().numpy())
    pred = np.concatenate(predictions)
    test_mse = float(np.median(np.mean((data.y - pred) ** 2, axis=1)))
    baseline = data.y - split.train.y.mean(axis=0, keepdims=True)
    test_baseline_mse = float(np.median(np.mean(baseline ** 2, axis=1)))
    tuned_rows.append({
        "dataset": task, "test_median_mse": test_mse,
        "test_baseline_median_mse": test_baseline_mse,
        "test_eta": test_baseline_mse / test_mse,
    })

tuned_df = pd.DataFrame(tuned_rows)
tuned_df.to_csv(OUT_ROOT / "new_tuned_eval.csv", index=False)


Seed set to 145
2026-07-29 13:27:09.889 | INFO     | omnixas.model.xasblock_regressor:load:265 - Loading model from /mnt/c/Users/anton/desktop/fullTrainingCopy072726/m3gnetAll8E2EUniversal/e2e_universal_seed42/notebook_shell_plateau_runs/weighted_shell_3_5/heads/universalXAS/All_FEFF/runs/plateau_lr5e-4_do010_seed44/best-model-epoch=273-val_median_mse=0.0021.ckpt
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
/home/anton/miniconda3/envs/omnixas/lib/python3.11/site-packages/lightning/pytorch/callbacks/model_checkpoint.py:881: Checkpoint directory /mnt/c/Users/anton/desktop/fullTrainingCopy072726/m3gnetAll8E2EUniversal/e2e_universal_seed42/notebook_shell_plateau_runs/weighted_shell_3_5/heads/tunedUniversalXAS/Ti_FEFF/runs/cosT500

Sanity Checking: |                                                                                | 0/? [00:00…

/home/anton/miniconda3/envs/omnixas/lib/python3.11/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=31` in the `DataLoader` to improve performance.
/home/anton/miniconda3/envs/omnixas/lib/python3.11/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=31` in the `DataLoader` to improve performance.


Training: |                                                                                       | 0/? [00:00…

Validation: |                                                                                     | 0/? [00:00…

...

Validation: |                                                                                     | 0/? [00:00…

Validation: |                                                                                     | 0/? [00:00…

Validation: |                                                                                     | 0/? [00:00…

Validation: |                                                                                     | 0/? [00:00…

2026-07-29 13:43:21.595 | INFO     | omnixas.model.xasblock_regressor:fit:246 - Best models saved at /mnt/c/Users/anton/desktop/fullTrainingCopy072726/m3gnetAll8E2EUniversal/e2e_universal_seed42/notebook_shell_plateau_runs/weighted_shell_3_5/heads/tunedUniversalXAS/Cu_FEFF/runs/cosT500_lr3e-4_do010_seed145/best-model-epoch=117-val_median_mse=0.0018.ckpt
2026-07-29 13:43:21.596 | INFO     | omnixas.model.xasblock_regressor:fit:247 - Best validation loss: 0.0027146635111421347


## 7. Final FEFF comparison table


In [7]:
previous_universal = pd.read_csv(PREVIOUS_RESULTS / "selected_universal_test_by_dataset.csv")[["dataset", "test_eta"]]
previous_universal = previous_universal.rename(columns={"test_eta": "previous_plateau_universal_test_eta"})
previous_tuned = pd.read_csv(PREVIOUS_RESULTS / "tuned_feff_selected_by_validation.csv")[["dataset", "test_eta"]]
previous_tuned = previous_tuned.rename(columns={"test_eta": "previous_plateau_tuned_test_eta"})
new_universal = universal_df[["dataset", "test_eta"]].rename(columns={"test_eta": "new_shell_universal_test_eta"})
new_tuned = tuned_df[["dataset", "test_eta"]].rename(columns={"test_eta": "new_shell_tuned_test_eta"})

comparison = pd.DataFrame({
    "dataset": FEFF_TASKS,
    "paper_universal_test_eta": [PAPER_FEFF_ETA[(task, "UniversalXAS")] for task in FEFF_TASKS],
    "paper_tuned_test_eta": [PAPER_FEFF_ETA[(task, "Tuned-UniversalXAS")] for task in FEFF_TASKS],
})
for frame in (previous_universal, previous_tuned, new_universal, new_tuned):
    if frame["dataset"].duplicated().any() or set(frame["dataset"]) != set(FEFF_TASKS):
        raise ValueError("Each comparison input must contain one row for every FEFF dataset")
    comparison = comparison.merge(frame, on="dataset", validate="one_to_one")

comparison = comparison[[
    "dataset",
    "paper_universal_test_eta",
    "paper_tuned_test_eta",
    "previous_plateau_universal_test_eta",
    "previous_plateau_tuned_test_eta",
    "new_shell_universal_test_eta",
    "new_shell_tuned_test_eta",
]]
comparison.to_csv(OUT_ROOT / "comparison_test_eta.csv", index=False)
display(comparison.style.format({column: "{:.3f}" for column in comparison.columns if column != "dataset"}))


,dataset,paper_universal_test_eta,paper_tuned_test_eta,previous_plateau_universal_test_eta,previous_plateau_tuned_test_eta,new_shell_universal_test_eta,new_shell_tuned_test_eta
0,Ti_FEFF,4.190,7.630,10.776,11.526,10.820,10.916
1,V_FEFF,5.190,9.220,11.170,11.751,11.447,11.990
2,Cr_FEFF,7.130,10.440,17.386,18.131,18.175,17.793
3,Mn_FEFF,13.150,29.810,25.504,33.966,28.387,35.108
4,Fe_FEFF,6.040,8.980,13.866,15.064,14.432,14.854
5,Co_FEFF,9.580,19.830,25.546,30.041,27.009,28.918
6,Ni_FEFF,6.430,11.210,15.251,16.242,16.097,14.862
7,Cu_FEFF,2.750,4.810,6.069,6.698,6.490,6.815
